# This code does everything needed for monthly progress report

In [8]:
### Start of the app

import gspread
import pandas as pd
from gspread_dataframe import set_with_dataframe
import numpy as np
import matplotlib.pyplot as plt
# 1. Authenticate using JSON key
gc = gspread.service_account(filename="python-project.json")

# 2. Open the Google Sheet
sh1 = gc.open_by_url("https://docs.google.com/spreadsheets/d/1T0dIc83PMS9PZSAHb3Qnz39UEmgakXskCMk-bVN7aUE/edit?gid=473559858#gid=473559858")
sh4 = gc.open_by_url("https://docs.google.com/spreadsheets/d/1T0dIc83PMS9PZSAHb3Qnz39UEmgakXskCMk-bVN7aUE/edit?gid=260563485#gid=260563485") #for publishing new tables
# 3. Select the specific worksheet (if there are many)
worksheet_calendly_interviews = sh1.worksheet("interviews")
worksheet_volunteer_success_applications = sh1.worksheet("volunteers")
worksheet_members_analytics = sh1.worksheet("slack")

#new sheet for publishing tables

In [6]:
worksheet_volunteer_success_applications.delete_rows(1, 2)

{'spreadsheetId': '1T0dIc83PMS9PZSAHb3Qnz39UEmgakXskCMk-bVN7aUE',
 'replies': [{}]}

In [9]:
df_interviews = pd.DataFrame(worksheet_calendly_interviews.get_all_records())
df_applications = pd.DataFrame(worksheet_volunteer_success_applications.get_all_records()) #All VSP volunteer records
df_analytics = pd.DataFrame(worksheet_members_analytics.get_all_records()) #A list of all Slack members and their activities


GSpreadException: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`

In [ ]:
df_applications.head()

,Name,Creation log,Email Address,YRES Email,Youth Advisor
0,Mariam Ali,"U+ HR Account Sep 11, 2025 1:30 PM",mariam.ali@yorkeducation.ca,,Bella He
1,Manusha Srikanthan,"Jess Scott Sep 11, 2025 1:45 PM",manushasrikanthan@gmail.com,,
2,Harvi Shah,"Jess Scott Sep 11, 2025 4:36 PM",harvishah2602@gmail.com,harvi.shah@yorkeducation.ca,Tiffany Ye
3,Dorsey Bangarh,"Jess Scott Sep 11, 2025 5:37 PM",dorseybangarh2004@gmail.com,dorsey.bangarh@yorkeducation.ca,
4,Pushti Ladani,"Jess Scott Sep 11, 2025 7:26 PM",pushti-kantilal.ladani@mohawkcollege.ca,pushti.ladani@yorkeducation.ca,


In [ ]:
df_interviews.head()

,Invitee Name,Invitee First Name,Invitee Last Name,Invitee Email,Event Type Name,Start Date & Time,End Date & Time,Event Created Date & Time,Canceled,Canceled By,Cancellation reason
0,Baani Singh,Baani,Singh,baanii.singh@gmail.com,ON: Volunteer Success Program,2025-03-11 4:45 PM,2025-03-11 5:00 PM,2025-02-18 4:11 PM,TRUE,Host,"Due to a scheduling conflict, we had to resche..."
1,Alisa Khvostiuk,Alisa,Khvostiuk,alisa.khvostiuk@gmail.com,ON: Volunteer Success Program,2025-03-03 3:00 PM,2025-03-03 3:15 PM,2025-02-20 3:39 PM,FALSE,,
2,Ryan Yang,Ryan,Yang,ryan.y2912@gmail.com,ON: Volunteer Success Program,2025-03-03 3:45 PM,2025-03-03 4:00 PM,2025-02-23 1:38 PM,FALSE,,
3,Baani Singh,Baani,Singh,baanii.singh@gmail.com,ON: Volunteer Success Program,2025-03-06 4:15 PM,2025-03-06 4:30 PM,2025-02-24 4:10 PM,FALSE,,
4,Mya Charlotte Wong,Mya,Charlotte Wong,vecchiamya@gmail.com,ON: Volunteer Success Program,2025-03-03 3:30 PM,2025-03-03 3:45 PM,2025-02-25 11:58 PM,FALSE,,


In [ ]:
df_analytics.head()

,Name,What I Do,Display name,Email,Account type,Account created (UTC),Claimed Date (UTC),Days active,Messages posted,Messages posted in channels,Reactions added,Last active (UTC)
0,(Vol. Leader) Victoria H.,,(Vol. Leader) Victoria H.,iivv.berry@gmail.com,Member,"Apr 14, 2023","Apr 14, 2023",0,0,0,0,
1,(Vol. Leader) Zainab Ahmed,,Zainab,zainab.ahmed@yorkeducation.ca,Member,"Apr 14, 2023","Apr 14, 2023",2,0,0,0,"Dec 30, 2025"
2,Aadam Lakhani,,Aadam Lakhani,aadam.lakhani@yorkeducation.ca,Member,"Sep 16, 2023","Sep 16, 2023",0,0,0,0,
3,Aadhya Sriram,,Aadhya Sriram,aadhya.sriram@yorkeducation.ca,Member,"Nov 8, 2023","Nov 8, 2023",0,0,0,0,"Aug 2, 2024"
4,Aakanksha,,Aakanksha,aakanksha.patel@yorkeducation.ca,Member,"Jul 25, 2025","Jul 25, 2025",0,0,0,0,"Nov 20, 2025"


1) Normalize emails

In [ ]:
df_applications["Email Address"] = df_applications["Email Address"].str.strip().str.lower()
df_applications["YRES Email"] = df_applications["YRES Email"].str.strip().str.lower()
df_analytics["Email"] = df_analytics["Email"].str.strip().str.lower()
df_interviews["Invitee Email"] = df_interviews["Invitee Email"].str.strip().str.lower()


2. Normalize date

In [ ]:
# Convert column Creation log to datetime for future visualizations
df_applications["datetime_str"] = df_applications["Creation log"].str.extract(r'([A-Z][a-z]{2}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2}\s+[AP]M)') #like Sep 26, 2025 3:07 AM
df_applications["date_joined_vps"] = pd.to_datetime(df_applications["datetime_str"])
df_applications["date_joined_vps"] #like 2025-09-26 03:07:00



/tmp/ipykernel_298484/1548341155.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_applications["date_joined_vps"] = pd.to_datetime(df_applications["datetime_str"])


0      2025-09-11 13:30:00
1      2025-09-11 13:45:00
2      2025-09-11 16:36:00
3      2025-09-11 17:37:00
4      2025-09-11 19:26:00
               ...        
1876   2025-12-30 16:58:00
1877   2025-12-31 11:41:00
1878   2025-12-31 13:10:00
1879   2025-12-31 13:19:00
1880   2025-12-31 14:48:00
Name: date_joined_vps, Length: 1881, dtype: datetime64[ns]

## Task 1. Table of students who did not have interview

In [ ]:
mask = ~df_applications["Email Address"].isin(df_interviews["Invitee Email"]) #email addresses that ARE NOT in interviewed emails
mask
df_no_interviews = df_applications[mask][["Name", "Email Address", "date_joined_vps", "YRES Email"]]
df_no_interviews = df_no_interviews.reset_index(drop=True) #reset index
df_no_interviews

,Name,Email Address,date_joined_vps,YRES Email
0,Mariam Ali,mariam.ali@yorkeducation.ca,2025-09-11 13:30:00,
1,Manusha Srikanthan,manushasrikanthan@gmail.com,2025-09-11 13:45:00,
2,Dorsey Bangarh,dorseybangarh2004@gmail.com,2025-09-11 17:37:00,dorsey.bangarh@yorkeducation.ca
3,Pushti Ladani,pushti-kantilal.ladani@mohawkcollege.ca,2025-09-11 19:26:00,pushti.ladani@yorkeducation.ca
4,Purti Ladani,purti1002@gmail.com,2025-09-11 19:29:00,purti.ladani@yorkeducation.ca
...,...,...,...,...
559,Kamsi Chinweokwu,princesskamsi18@gmail.com,2025-12-28 22:16:00,
560,Dushikaa Umasangaran,,2025-12-29 21:51:00,
561,Sarah Hwang,sarahhwang11@gmail.com,2025-12-30 16:58:00,
562,Ashley Ramnarace,ashley0252@hotmail.com,2025-12-31 13:10:00,


In [ ]:
# Add new worksheet (tab)
ws = sh4.add_worksheet(title="No Interview", rows="1000", cols="20")

# Upload DataFrame
set_with_dataframe(ws, df_no_interviews, include_index=False)

# Save it to excel table too
# df_no_interviews.to_excel('tables/no_interviews_table.xlsx', index=False)

APIError: APIError: [400]: Invalid requests[0].addSheet: A sheet with the name "No Interview" already exists. Please enter another name.

## Task 2. Who haven't been invited to Slack

In [ ]:
mask = df_applications["Youth Advisor"] !="" #those who have youth advisor
df_applications_have_advisor= df_applications[mask]
df_applications_have_advisor

,Name,Creation log,Email Address,YRES Email,Youth Advisor,datetime_str,date_joined_vps
0,Mariam Ali,"U+ HR Account Sep 11, 2025 1:30 PM",mariam.ali@yorkeducation.ca,,Bella He,"Sep 11, 2025 1:30 PM",2025-09-11 13:30:00
2,Harvi Shah,"Jess Scott Sep 11, 2025 4:36 PM",harvishah2602@gmail.com,harvi.shah@yorkeducation.ca,Tiffany Ye,"Sep 11, 2025 4:36 PM",2025-09-11 16:36:00
6,Harjot Singh,"Jess Scott Sep 11, 2025 8:52 PM",singhharjot1312@gmail.com,harjot.singh@yorkeducation.ca,Aanushan Elangoban,"Sep 11, 2025 8:52 PM",2025-09-11 20:52:00
8,JeeHu Choi,"Jess Scott Sep 11, 2025 8:55 PM",jiwhotwin@gmail.com,jeehu.choi@yorkeducation.ca,Bella He,"Sep 11, 2025 8:55 PM",2025-09-11 20:55:00
9,Taranveer Arora,"Jess Scott Sep 11, 2025 9:32 PM",taransarora@gmail.com,taranveer.arora@yorkeducation.ca,Bryna McGarrigle,"Sep 11, 2025 9:32 PM",2025-09-11 21:32:00
...,...,...,...,...,...,...,...
1827,Farsam Shirazi,"Jess Scott Dec 14, 2025 7:13 PM",farsam.shirazi@gmail.com,,Aanushan Elangoban,"Dec 14, 2025 7:13 PM",2025-12-14 19:13:00
1828,Chris Xu,"Jess Scott Dec 14, 2025 10:00 PM",chrisxu607@gmail.com,chris.xu@yorkeducation.ca,Aanushan Elangoban,"Dec 14, 2025 10:00 PM",2025-12-14 22:00:00
1832,Jiayuan(Jack) Zheng,"Jess Scott Dec 16, 2025 4:27 PM",zhengjiayuan23@gmail.com,jiayuan.jack.zheng@yorkeducation.ca,Jing Zhang,"Dec 16, 2025 4:27 PM",2025-12-16 16:27:00
1833,Anthony Bosco,"Jess Scott Dec 16, 2025 6:45 PM",anthony.bosco2609@gmail.com,anthony.bosco@yorkeducation.ca,Aanushan Elangoban,"Dec 16, 2025 6:45 PM",2025-12-16 18:45:00


In [ ]:
mask = ~df_applications_have_advisor["YRES Email"].isin(df_analytics["Email"])
df_not_invited = df_applications_have_advisor[mask][["Name", "Email Address", "YRES Email", "Youth Advisor", "date_joined_vps"]]
df_not_invited = df_not_invited.reset_index(drop=True)
df_not_invited = df_not_invited.dropna(subset=['YRES Email'])
df_not_invited

,Name,Email Address,YRES Email,Youth Advisor,date_joined_vps
0,Mariam Ali,mariam.ali@yorkeducation.ca,,Bella He,2025-09-11 13:30:00
1,Olina Zheng,olinazheng10@gmail.com,olina.zheng@yorkeducation.ca,Bella He,2025-09-13 15:39:00
2,Shoaib Akthar Shaik,shaikshoaib13251@gmail.com,,Bryna McGarrigle,2025-09-24 17:13:00
3,Kayla Orellana,kaylaorellana33@gmail.com,kayla.orellana@yorkeducation.ca,Aanushan Elangoban,2025-09-26 07:13:00
4,Carter Wang,ziyao_wang81@hotmail.com,,Damon Tran,2025-09-28 22:45:00
...,...,...,...,...,...
121,Akilan Pathmanathan,akilan9905@hotmail.com,akilan.pathmanathan@yorkeducation.ca,Damon Tran,2025-11-25 19:31:00
122,Kashish Dhanani,dhananikashish33@gmail.com,kashish.dhanani@yorkeducation.ca,Aanushan Elangoban,2025-12-04 18:49:00
123,Samson Maveli Mathews,samsonmmathews@gmail.com,samson.mathews@yorkeducation.ca,Aanushan Elangoban,2025-12-04 21:16:00
124,Prasha Maskey,maskeyprasha92@gmail.com,,Bella He,2025-10-12 18:57:00


In [ ]:
# Add new worksheet (tab)
ws = sh4.add_worksheet(title="No Slack added", rows="1000", cols="20")

# Upload DataFrame
set_with_dataframe(ws, df_not_invited, include_index=False)

# Save it to excel table too
# df_not_invited.to_excel('tables/no_slack_table.xlsx', index=False)

## Task 3. Who are invited to Slack but never joined?

In [ ]:
mask = df_applications["YRES Email"] !="" #those who have youth advisor
df_applications_yres_not_null= df_applications[mask]
df_applications_yres_not_null

,Name,Creation log,Email Address,YRES Email,Youth Advisor,datetime_str,date_joined_vps
2,Harvi Shah,"Jess Scott Sep 11, 2025 4:36 PM",harvishah2602@gmail.com,harvi.shah@yorkeducation.ca,Tiffany Ye,"Sep 11, 2025 4:36 PM",2025-09-11 16:36:00
3,Dorsey Bangarh,"Jess Scott Sep 11, 2025 5:37 PM",dorseybangarh2004@gmail.com,dorsey.bangarh@yorkeducation.ca,,"Sep 11, 2025 5:37 PM",2025-09-11 17:37:00
4,Pushti Ladani,"Jess Scott Sep 11, 2025 7:26 PM",pushti-kantilal.ladani@mohawkcollege.ca,pushti.ladani@yorkeducation.ca,,"Sep 11, 2025 7:26 PM",2025-09-11 19:26:00
5,Purti Ladani,"Jess Scott Sep 11, 2025 7:29 PM",purti1002@gmail.com,purti.ladani@yorkeducation.ca,,"Sep 11, 2025 7:29 PM",2025-09-11 19:29:00
6,Harjot Singh,"Jess Scott Sep 11, 2025 8:52 PM",singhharjot1312@gmail.com,harjot.singh@yorkeducation.ca,Aanushan Elangoban,"Sep 11, 2025 8:52 PM",2025-09-11 20:52:00
...,...,...,...,...,...,...,...
1826,Neo Lui,"Jess Scott Dec 14, 2025 11:19 AM",matchman692@gmail.com,neo.lui@yorkeducation.ca,Jing Zhang,"Dec 14, 2025 11:19 AM",2025-12-14 11:19:00
1828,Chris Xu,"Jess Scott Dec 14, 2025 10:00 PM",chrisxu607@gmail.com,chris.xu@yorkeducation.ca,Aanushan Elangoban,"Dec 14, 2025 10:00 PM",2025-12-14 22:00:00
1832,Jiayuan(Jack) Zheng,"Jess Scott Dec 16, 2025 4:27 PM",zhengjiayuan23@gmail.com,jiayuan.jack.zheng@yorkeducation.ca,Jing Zhang,"Dec 16, 2025 4:27 PM",2025-12-16 16:27:00
1833,Anthony Bosco,"Jess Scott Dec 16, 2025 6:45 PM",anthony.bosco2609@gmail.com,anthony.bosco@yorkeducation.ca,Aanushan Elangoban,"Dec 16, 2025 6:45 PM",2025-12-16 18:45:00


In [ ]:
mask = df_analytics["Account type"]=="Invited Member"
df_analytics_invited = df_analytics[mask]
mask = df_applications_yres_not_null["YRES Email"].isin(df_analytics_invited["Email"])
df_not_joined = df_applications_yres_not_null[mask][["Name", "Email Address", "YRES Email", "Youth Advisor", "date_joined_vps"]].reset_index(drop=True)
email_to_type = dict(
    zip(df_analytics_invited["Email"], df_analytics_invited["Account type"])
)

df_not_joined["Account type"] = df_not_joined["YRES Email"].map(email_to_type)

df_not_joined

,Name,Email Address,YRES Email,Youth Advisor,date_joined_vps,Account type
0,Chik Fu Fung,chikf666@gmail.com,chik.fu.fung@yorkeducation.ca,Chelzy Evangelista,2025-09-14 00:44:00,Invited Member
1,Samir Altakhin,takhinsamir@gmail.com,samir.altakhin@yorkeducation.ca,Jing Zhang,2025-09-17 12:23:00,Invited Member
2,Wahenoor Kaur,wahenoork2009@gmail.com,wahenoor.kaur@yorkeducation.ca,Chelzy Evangelista,2025-09-15 23:32:00,Invited Member
3,Safaet Ahmed,safaetahmed.00@gmail.com,safaet.ahmed@yorkeducation.ca,Jing Zhang,2025-09-16 07:18:00,Invited Member
4,Marly Astudillo,jamileth.astudillo.orellana@gmail.com,marly.astudillo@yorkeducation.ca,Tiffany Ye,2025-09-17 23:57:00,Invited Member
...,...,...,...,...,...,...
90,Mehakpreet kaur,mehakpreetkaur086@gmail.com,mehakpreet.kaur2@yorkeducation.ca,Tiffany Ye,2025-09-14 23:04:00,Invited Member
91,Emmanuel Inebode,emmainebode80@gmail.com,emmanuel.inebode@yorkeducation.ca,Jing Zhang,2025-11-02 11:24:00,Invited Member
92,Emmanuel Inebode,emmainebode80@gmail.com,emmanuel.inebode@yorkeducation.ca,Jing Zhang,2025-11-02 11:30:00,Invited Member
93,Emmanuel Inebode,emmainebode80@gmail.com,emmanuel.inebode@yorkeducation.ca,Jing Zhang,2025-11-02 11:36:00,Invited Member


In [ ]:
# Add new worksheet (tab)
ws = sh4.add_worksheet(title="Haven't join Slack", rows="1000", cols="20")

# Upload DataFrame
set_with_dataframe(ws, df_not_joined, include_index=False)

# Save it to excel table too
df_not_joined.to_excel('tables/no_joined_table.xlsx', index=False)

OSError: Cannot save file into a non-existent directory: 'tables'

## Task 4. Haven't been active since September 30.

In [ ]:
df_analytics["last_active"] = pd.to_datetime(
    df_analytics["Last active (UTC)"], errors="coerce"
).dt.date

# Inactive since before 2025-10-01
cutoff = pd.to_datetime("2025-10-31").date()

df_slack_inactive = df_analytics[df_analytics["last_active"] < cutoff].copy()
# Slack: normalize Email (YRES email)
df_slack_inactive["Email_norm"] = (
    df_slack_inactive["Email"].astype(str).str.strip().str.lower()
)

inactive_yres_emails = set(df_slack_inactive["Email_norm"])

df_not_active = df_applications_yres_not_null[
    df_applications_yres_not_null["YRES Email"].isin(inactive_yres_emails)
][[
    "Name",
    "Email Address",
    "YRES Email",
    "Youth Advisor",
    "date_joined_vps",
]].reset_index(drop=True)

email_to_last_active = dict(
    zip(df_slack_inactive["Email_norm"], df_slack_inactive["last_active"])
)

df_not_active["last_active"] = df_not_active["YRES Email"].map(email_to_last_active)
df_not_active = df_not_active[[
    "Name",
    "Email Address",
    "YRES Email",
    "Youth Advisor",
    "date_joined_vps",
    "last_active"
]]
df_not_active = df_not_active.sort_values(by='last_active').reset_index(drop=True)
df_not_active

,Name,Email Address,YRES Email,Youth Advisor,date_joined_vps,last_active
0,Crystal Chuong,chuongcrystal82@gmail.com,crystal.chuong@yorkeducation.ca,,2025-04-16 13:12:00,2024-10-27
1,Adrian Anton Dominic,adrian365545@gmail.com,adrian.anton-dominic@yorkeducation.ca,,2025-04-12 15:35:00,2024-12-12
2,Michael Chu,michaelchuc123@gmail.com,michael.chu@yorkeducation.ca,,2025-04-09 16:39:00,2025-02-06
3,Sara Mahfouz,mahfouzsara1@gmail.com,sara.mahfouz@yorkeducation.ca,,2025-05-07 22:11:00,2025-02-24
4,Kathir Paraneetharan,kathirparani2@gmail.com,kathir.paraneetharan@yorkeducation.ca,,2025-03-11 10:41:00,2025-03-06
...,...,...,...,...,...,...
428,Neomi Garbo,neomiaishagarbo@gmail.com,neomi.garbo@yorkeducation.ca,Bella He,2025-07-15 17:31:00,2025-10-30
429,Yin-Wen (Ella) Tsai,yinwentsai@gmail.com,yin-wen.tsai@yorkeducation.ca,Chelzy Evangelista,2025-08-16 04:19:00,2025-10-30
430,Aleeza Usman,aleezausman234@gmail.com,aleeza.usman@yorkeducation.ca,Damon Tran,2025-05-27 18:39:00,2025-10-30
431,Husna Ahmad,husna.a0216@gmail.com,husna.ahmad@yorkeducation.ca,Aanushan Elangoban,2025-08-07 17:53:00,2025-10-30


In [ ]:
# Add new worksheet (tab)
ws = sh4.add_worksheet(title="Not active", rows="1000", cols="20")

# Upload DataFrame
set_with_dataframe(ws, df_not_active, include_index=False)

# Save it to excel table too
# df_not_active.to_excel('tables/not_active_table.xlsx', index=False)

## Task 5. Messages posted by VSP volunteers in October

In [ ]:
# Parse datetime
df_analytics["Last active (UTC)"] = pd.to_datetime(
    df_analytics["Last active (UTC)"], errors="coerce"
)

# Now merge
merged = pd.merge(
    df_applications_have_advisor,
    df_analytics,
    how="inner"   # IMPORTANT
)

df_messages = merged[[
    "Name",
    "YRES Email",
    "date_joined_vps",
    "Last active (UTC)",
    "Messages posted",
    "Messages posted in channels",
    "Reactions added"
]].reset_index(drop=True)


message_cols = [
    "Messages posted",
    "Messages posted in channels",
    "Reactions added"
]
for col in message_cols:
    df_messages[col] = (
        df_messages[col]
        .fillna(0)    # replace NaN with 0
        .astype(int)  # convert float → int
    )

df_messages["Last active (UTC)"] = df_messages["Last active (UTC)"].fillna("")
df_messages = df_messages.sort_values(by="Name").reset_index(drop=True)
df_messages

,Name,YRES Email,date_joined_vps,Last active (UTC),Messages posted,Messages posted in channels,Reactions added
0,ANDI DONG,andi.dong@yorkeducation.ca,2025-10-10 18:20:00,2025-10-30,0,0,0
1,Aaradhya Atul Yeginwar,aaradhya@yorkeducation.ca,2025-05-28 00:45:00,2025-10-28,0,0,0
2,Aaran Rajkanth,aaranrajkanth@yorkeducation.ca,2025-06-13 19:15:00,2025-12-19,12,0,0
3,Aaron Zhang,aaron.zhang@yorkeducation.ca,2025-03-10 10:15:00,2025-11-07,0,0,0
4,Aarthi Thiravidamani,aarthi.thiravidamani@yorkeducation.ca,2025-12-08 16:22:00,2026-01-02,6,0,0
...,...,...,...,...,...,...,...
696,Zoha Shaukat,zoha.shaukat@yorkeducation.ca,2025-05-25 19:01:00,2025-10-31,0,0,0
697,Zohra Durani,zohra.durani@yorkeducation.ca,2025-09-22 19:48:00,2025-10-21,0,0,0
698,Zoya Hasan,zoya.hasan@yorkeducation.ca,2025-04-26 11:03:00,2025-10-29,0,0,0
699,keegan keene,keegan.keene@yorkeducation.ca,2025-11-07 12:55:00,2025-12-04,1,0,0


In [ ]:
# Add new worksheet (tab)
ws = sh4.add_worksheet(title="Employee messages", rows="2000", cols="0")

# Upload DataFrame
set_with_dataframe(ws, df_messages, include_index=False)

# Save it to excel table too
# df_messages.to_excel('tables/employee_messages_table.xlsx', index=False)